<a href="https://colab.research.google.com/github/shreyoshi2304/Dataset-Analysis-/blob/main/Youtube_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Originally done in Anaconda; pasted here

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(r"C:\Users\Shreyoshi\OneDrive\Desktop\DsResearch\Media and Technology\Media and Technology\Global YouTube Statistics.csv",encoding ='ISO-8859-1')

#Extracting & Analyzing the overall data summary
print(df.info())
print(df.describe())
df = df.drop_duplicates()

print(df.shape)
'Total data points is 995'

print(df.columns) #All Columns


#Removing Redundant Columns

df.drop(columns=["Title"], inplace = True)
'''Title and Youtuber are redundant'''
df.drop(columns=["Country of origin","Abbreviation"], inplace = True)
'''Country of origin , Country & Abbreviation are redundant'''

print(df.columns)

print(df.isnull().sum())

# Population
missing_population = df[df['Population'].isnull()]
print(missing_population['Country'].value_counts())

# Gross tertiary education enrollment (%)
missing_edu = df[df['Gross tertiary education enrollment (%)'].isnull()]
print(missing_edu['Country'].value_counts(dropna=False))


# Unemployment rate
missing_unemp = df[df['Unemployment rate'].isnull()]
print(missing_unemp['Country'].value_counts(dropna=False))

#Filling the information on the country Andorra
df.loc[(df['Country'] == 'Andorra') & (df['Population'].isnull()), 'Population'] = 82904

df.loc[(df['Country'] == 'Andorra') &
       (df['Gross tertiary education enrollment (%)'].isnull()),
       'Gross tertiary education enrollment (%)'] = 64.28

df.loc[(df['Country'] == 'Andorra') &
       (df['Unemployment rate'].isnull()),
       'Unemployment rate'] = 1.6
################################################################################################################
df.dropna(subset=['subscribers'], inplace= True)

top_10_channels = df.sort_values(by='subscribers', ascending=False).head(10)

# Display the top 10
print(top_10_channels[['Youtuber', 'subscribers', 'Country']])

plt.figure(figsize=(14, 8))
sns.barplot(data=top_10_channels, x='Youtuber', y='subscribers', palette='viridis')

plt.title('Top 10 YouTube Channels by Subscribers', fontsize=16, weight='bold')
plt.xlabel('YouTube Channel', fontsize=14)
plt.ylabel('Number of Subscribers', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
##############################################################################################################
df['category']= df['category'].fillna(df['category'].mode()[0])

# Group by 'category' and calculate average subscribers
avg_subs_by_category = df.groupby('category')['subscribers'].mean().sort_values(ascending=False)

# Display the category with highest average subscribers
top_category = avg_subs_by_category.idxmax()
top_avg_subs = avg_subs_by_category.max()

print(f"The category with the highest average subscribers is: {top_category} with an average of {top_avg_subs:.2f} subscribers.")


plt.figure(figsize=(12,6))
sns.barplot(x=avg_subs_by_category.values, y=avg_subs_by_category.index, palette='coolwarm')
plt.xlabel('Average Subscribers')
plt.ylabel('Category')
plt.title('Average Subscribers by YouTube Category')
plt.show()

##################################################################################################################
# Group by 'category' and calculate average uploads
avg_uploads_by_category = df.groupby('category')['uploads'].mean().sort_values(ascending=False)

# Display the average uploads per category
print(avg_uploads_by_category)

plt.figure(figsize=(12,6))
sns.barplot(x=avg_uploads_by_category.values, y=avg_uploads_by_category.index, palette='mako')
plt.xlabel('Average Number of Uploads')
plt.ylabel('YouTube Category')
plt.title('Average Uploads per Channel by Category')
plt.show()
##################################################################################################################
df['Country'] = df['Country'].fillna(df['Country'].mode()[0])
top_countries = df['Country'].value_counts().sort_values(ascending=False).head(5)
print(top_countries)

plt.figure(figsize=(12,6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='Set2')
plt.xlabel('Number of YouTube Channels')
plt.ylabel('Country')
plt.title('Top 10 Countries by Number of YouTube Channels')
plt.show()
##################################################################################################################
df['channel_type'] = df['channel_type'].fillna(df['channel_type'].mode()[0])

# Create a pivot table: Counts of channel types within each category
channel_type_distribution = pd.pivot_table(df,
                                           index='category',
                                           columns='channel_type',
                                           values='Youtuber',
                                           aggfunc='count',
                                           fill_value=0)

print(channel_type_distribution)

channel_type_distribution.plot(kind='bar', stacked=True, figsize=(14,8), colormap='tab20c')

plt.title('Distribution of Channel Types Across Categories', fontsize=16)
plt.xlabel('YouTube Category', fontsize=12)
plt.ylabel('Number of Channels', fontsize=12)
plt.legend(title='Channel Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
#################################################################################################
numeric_df = df[['subscribers', 'video views']].copy()
corr_matrix = numeric_df.corr()
plt.figure(figsize=(12,6))
sns.heatmap(corr_matrix , annot = True, cmap = 'PuBuGn' , fmt = '.2f')
plt.title('Correlation between Subscribers and Video Views', fontsize=14)
plt.show()

sns.pairplot(numeric_df)
plt.show()
###################################################################################################
df['average_monthly_earnings'] = (df['lowest_monthly_earnings'] + df['highest_monthly_earnings']) / 2
category_earnings = df.groupby('category')['average_monthly_earnings'].mean().sort_values(ascending=False)
plt.figure(figsize=(12,6))
sns.barplot(x=category_earnings.values, y=category_earnings.index, palette='crest')
plt.xlabel('Average Monthly Earnings')
plt.ylabel('YouTube Category')
plt.title('Average Monthly Earnings by YouTube Category', fontsize=16)
plt.tight_layout()
plt.show()
################################################################################################################################
df['subscribers_for_last_30_days'] = df['subscribers_for_last_30_days'].fillna(df['subscribers_for_last_30_days'].median())
# Basic Summary Statistics
print("\n Summary Statistics for Subscribers Gained/Lost (Last 30 Days):\n")
print(df['subscribers_for_last_30_days'].describe())

# Distribution Plot (Growth vs. Loss)
plt.figure(figsize=(10,6))
sns.histplot(df['subscribers_for_last_30_days'], bins=20, kde=True, color='skyblue')
plt.xlabel('Subscribers Gained/Lost in Last 30 Days')
plt.title('Distribution of Subscriber Changes (Last 30 Days)')
plt.tight_layout()
plt.show()

#Top 10 Gainers
top_gainers = df.sort_values(by='subscribers_for_last_30_days', ascending=False).head(10)
print("\n Top 10 Channels by Subscriber Gain (Last 30 Days):\n")
print(top_gainers[['Youtuber', 'subscribers_for_last_30_days', 'Country']])

# Plot Top 10 Gainers
plt.figure(figsize=(12,6))
sns.barplot(x=top_gainers['Youtuber'], y=top_gainers['subscribers_for_last_30_days'], palette='Greens_r')
plt.title('Top 10 Channels by Subscriber Gain (Last 30 Days)', fontsize=16)
plt.xlabel('Channel Name')
plt.ylabel('Subscribers Gained')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#Top 10 Losers
top_losers = df.sort_values(by='subscribers_for_last_30_days').head(10)
print("\n Top 10 Channels by Subscriber Loss (Last 30 Days):\n")
print(top_losers[['Youtuber', 'subscribers_for_last_30_days', 'Country']])

# Plot Top 10 Losers
plt.figure(figsize=(12,6))
sns.barplot(x=top_losers['Youtuber'], y=top_losers['subscribers_for_last_30_days'], palette='Reds')
plt.title('Top 10 Channels by Subscriber Loss (Last 30 Days)', fontsize=16)
plt.xlabel('Channel Name')
plt.ylabel('Subscribers Lost')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


#Category-wise Average Subscriber Change
category_trend = df.groupby('category')['subscribers_for_last_30_days'].mean().sort_values(ascending=False)

plt.figure(figsize=(12,6))
sns.barplot(x=category_trend.values, y=category_trend.index, palette='crest')
plt.xlabel('Average Subscribers Gained/Lost (Last 30 Days)')
plt.ylabel('Category')
plt.title('Average Subscriber Change by Category')
plt.tight_layout()
plt.show()

#Country-wise Average Subscriber Change (Top 10 Countries)
country_trend = df.groupby('Country')['subscribers_for_last_30_days'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=country_trend.values, y=country_trend.index, palette='rocket')
plt.xlabel('Average Subscribers Gained (Last 30 Days)')
plt.ylabel('Country')
plt.title('Top Countries by Average Subscriber Gain (Last 30 Days)')
plt.tight_layout()
plt.show()

################################################################################################################
df['average_yearly_earnings'] = (df['lowest_yearly_earnings'] + df['highest_yearly_earnings']) / 2
# Calculate IQR
Q1 = df['average_yearly_earnings'].quantile(0.25)
Q3 = df['average_yearly_earnings'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Find outliers
outliers_income = df[(df['average_yearly_earnings'] < lower_bound) | (df['average_yearly_earnings'] > upper_bound)]

print("Number of Outliers in Average Yearly Earnings:", outliers_income.shape[0])
print(outliers_income[['Youtuber', 'average_yearly_earnings']].sort_values(by='average_yearly_earnings', ascending=False))

plt.figure(figsize=(12,6))
sns.boxplot(y=df['average_yearly_earnings'], color='skyblue')
plt.title('Boxplot of Average Yearly Earnings (Outlier Detection)')
plt.xlabel('Average Yearly Earnings')
plt.tight_layout()
plt.show()

# ======================= LOWEST YEARLY EARNINGS ===========================
Q1_low = df['lowest_yearly_earnings'].quantile(0.25)
Q3_low = df['lowest_yearly_earnings'].quantile(0.75)
IQR_low = Q3_low - Q1_low

lower_bound_low = Q1_low - 1.5 * IQR_low
upper_bound_low = Q3_low + 1.5 * IQR_low

outliers_low = df[(df['lowest_yearly_earnings'] < lower_bound_low) | (df['lowest_yearly_earnings'] > upper_bound_low)]

print("\n Outliers in LOWEST Yearly Earnings:", outliers_low.shape[0])
print(outliers_low[['Youtuber', 'lowest_yearly_earnings']].sort_values(by='lowest_yearly_earnings', ascending=False))

plt.figure(figsize=(12,6))
sns.boxplot(y=df['lowest_yearly_earnings'], color='lightcoral')
plt.title('Boxplot of LOWEST Yearly Earnings (Outlier Detection)', fontsize=14)
plt.ylabel('Lowest Yearly Earnings')
plt.tight_layout()
plt.show()


# ======================= HIGHEST YEARLY EARNINGS ===========================
Q1_high = df['highest_yearly_earnings'].quantile(0.25)
Q3_high = df['highest_yearly_earnings'].quantile(0.75)
IQR_high = Q3_high - Q1_high

lower_bound_high = Q1_high - 1.5 * IQR_high
upper_bound_high = Q3_high + 1.5 * IQR_high

outliers_high = df[(df['highest_yearly_earnings'] < lower_bound_high) | (df['highest_yearly_earnings'] > upper_bound_high)]

print("\n Outliers in HIGHEST Yearly Earnings:", outliers_high.shape[0])
print(outliers_high[['Youtuber', 'highest_yearly_earnings']].sort_values(by='highest_yearly_earnings', ascending=False))

plt.figure(figsize=(12,6))
sns.boxplot(y=df['highest_yearly_earnings'], color='mediumseagreen')
plt.title('Boxplot of HIGHEST Yearly Earnings (Outlier Detection)', fontsize=14)
plt.ylabel('Highest Yearly Earnings')
plt.tight_layout()
plt.show()

###########################################################################################################

# Define mapping for month names to numbers
month_mapping = {
    'Jan': 1, 'January': 1,
    'Feb': 2, 'February': 2,
    'Mar': 3, 'March': 3,
    'Apr': 4, 'April': 4,
    'May': 5,
    'Jun': 6, 'June': 6,
    'Jul': 7, 'July': 7,
    'Aug': 8, 'August': 8,
    'Sep': 9, 'Sept': 9, 'September': 9,
    'Oct': 10, 'October': 10,
    'Nov': 11, 'November': 11,
    'Dec': 12, 'December': 12
}

# Apply mapping where needed
df['created_month'] = df['created_month'].replace(month_mapping)

# Fill missing values
df['created_date'] = df['created_date'].fillna(df['created_date'].median())
df['created_month'] = df['created_month'].fillna(df['created_month'].mode()[0])
df['created_year'] = df['created_year'].fillna(df['created_year'].mode()[0])

df['created_year'] = df['created_year'].astype(int)
df['created_month'] = df['created_month'].astype(int)
df['created_date'] = df['created_date'].astype(int)

# Correct datetime creation WITHOUT redundant fillna
df['created_full_date'] = pd.to_datetime(dict(
    year=df['created_year'],
    month=df['created_month'],
    day=df['created_date']
), errors='coerce')

# Remove rows with invalid dates (NaT)
df = df[df['created_full_date'].notnull()]

# Create Year-Month
df['YearMonth'] = df['created_full_date'].dt.to_period('M')

# Check: If YearMonth has data
print(df['YearMonth'].value_counts().head())

# Plot
channels_per_month = df['YearMonth'].value_counts().sort_index()

plt.figure(figsize=(16,8))
channels_per_month.plot(kind='line', marker='o', color='teal')
plt.title('YouTube Channel Creation Trend Over Time (Monthly)', fontsize=16)
plt.xlabel('Year-Month')
plt.ylabel('Number of Channels Created')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

creation_counts = df['YearMonth'].value_counts().sort_values(ascending=False)
top10_creation_times = creation_counts.head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=top10_creation_times.index.astype(str), y=top10_creation_times.values, palette='viridis')

plt.title('Top 10 Most Frequent Times of YouTube Channel Creation', fontsize=16)
plt.xlabel('Year-Month', fontsize=12)
plt.ylabel('Number of Channels Created', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
############################################################################################################
# Step 1: Group by Country and Count YouTube Channels
channel_counts = df.groupby('Country')['Youtuber'].count().reset_index()
channel_counts.rename(columns={'Youtuber': 'Number_of_Channels'}, inplace=True)

# Step 2: Get Tertiary Education Data (drop duplicates to avoid repeats)
edu_data = df[['Country', 'Gross tertiary education enrollment (%)']].drop_duplicates()

# Step 3: Merge Channel Counts with Education Data
channel_edu_df = pd.merge(channel_counts, edu_data, on='Country', how='left')

# Step 4: Drop countries where education data is missing
channel_edu_df = channel_edu_df.dropna(subset=['Gross tertiary education enrollment (%)'])

# Step 5: Plot Scatter Plot
plt.figure(figsize=(12,7))
sns.scatterplot(data=channel_edu_df,
                x='Gross tertiary education enrollment (%)',
                y='Number_of_Channels',
                hue='Country',
                s=100, palette='tab10', legend=False)

plt.title('Relationship between Tertiary Education Enrollment and Number of YouTube Channels', fontsize=16)
plt.xlabel('Gross Tertiary Education Enrollment (%)', fontsize=12)
plt.ylabel('Number of YouTube Channels', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Step 6: Calculate Pearson Correlation
correlation = channel_edu_df['Gross tertiary education enrollment (%)'].corr(channel_edu_df['Number_of_Channels'])
print(f"Correlation between Tertiary Education Enrollment and YouTube Channel Count: {correlation:.2f}")

# Step 2: Compute Correlation Matrix
corr_matrix = channel_edu_df[['Number_of_Channels',
                                'Gross tertiary education enrollment (%)'
                                ]].corr()

# Step 3: Plot Heatmap
plt.figure(figsize=(10,6))
sns.heatmap(corr_matrix, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5, square=True)

plt.title('Correlation Heatmap: Education vs YouTube Channels', fontsize=14)
plt.tight_layout()
plt.show()

############################################################################################################

unempdf = df.dropna(subset=['Unemployment rate'])
# Step 2: Merge with Unemployment Data
unemp_data = unempdf[['Country', 'Unemployment rate']].drop_duplicates()
channel_unemp_df = pd.merge(channel_counts, unemp_data, on='Country', how='left')

# Step 3: Get Top 10 Countries by Number of Channels
top10_countries = channel_unemp_df.sort_values(by='Number_of_Channels', ascending=False).head(10)

# Step 4: Plot Unemployment Rates for Top 10 Countries
plt.figure(figsize=(12,7))
sns.barplot(data=top10_countries,
            x='Country',
            y='Unemployment rate',
            palette='mako')

plt.title('Unemployment Rate in Top 10 Countries by YouTube Channel Count', fontsize=16)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Unemployment Rate (%)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

#############################################################################################################

urbandf = df.dropna(subset = ['Urban_population'])
urbangroupdf = urbandf.groupby('Country')['Urban_population'].mean().reset_index()

urban_count = pd.merge(urbangroupdf , channel_counts , on ="Country" , how = "left")
top15_countries = urban_count.sort_values(by='Number_of_Channels', ascending=False).head(15)
plt.figure(figsize=(12,8))
sns.barplot( data = top15_countries,
            x= "Country",
            y="Urban_population",
            palette = 'mako')
plt.title('Urban Population in Top 15 Countries by YouTube Channel Count', fontsize=16)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Urban Population', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

bottom15_countries = urban_count.sort_values(by='Number_of_Channels', ascending=False).tail(15)
plt.figure(figsize=(12,8))
sns.barplot( data = bottom15_countries,
            x= "Country",
            y="Urban_population",
            palette = 'mako')
plt.title('Urban Population in Bottom 15 Countries by YouTube Channel Count', fontsize=16)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Urban Population', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

####################################################################################################

import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'browser'  # ← This line is key for Spyder

# Remove rows with missing coordinates
geo_df = df.dropna(subset=['Latitude', 'Longitude'])
print(geo_df.shape)
# Plot
fig = px.scatter_geo(
    geo_df,
    lat='Latitude',
    lon='Longitude',
    hover_name='Youtuber',
    color='Country',
    title=' Global Distribution of YouTube Channels',
    projection='natural earth',
    opacity=0.7
)

fig.update_layout(height=600, margin={"r":0,"t":40,"l":0,"b":0})
fig.show()


##############################################################################################

# Group by Country: total subscribers and population
subs_pop = df.groupby('Country').agg({
    'subscribers': 'sum',
    'Population': 'first'  # Assuming Population is the same for each country
}).dropna()

# Calculate correlation
corr = subs_pop['subscribers'].corr(subs_pop['Population'])
print(f" Correlation between total subscribers and population: {corr:.2f}")

# Optional: Scatter plot
plt.figure(figsize=(10,6))
sns.scatterplot(data=subs_pop, x='Population', y='subscribers', s=100, alpha=0.7)
plt.title('Subscribers vs Population by Country')
plt.xlabel('Population')
plt.ylabel('Total Subscribers')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='YlGnBu', fmt='.2f')

plt.title('Correlation Heatmap: Subscribers vs Population', fontsize=14)
plt.tight_layout()
plt.show()

#################################################################################################

top_countries = df['Country'].value_counts().head(10).index
top_countries_df = df[df['Country'].isin(top_countries)].drop_duplicates(subset=['Country'])

plt.figure(figsize=(12,7))
sns.barplot(data=top_countries_df, x='Country', y='Population', palette='rocket')

plt.title('Top 10 Countries by Number of YouTube Channels vs Population')
plt.xlabel('Country')
plt.ylabel('Population')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#########################################################################################################

subs_unemp_df = df[['Country', 'subscribers_for_last_30_days', 'Unemployment rate']].dropna()

subs_unemp_agg = subs_unemp_df.groupby('Country').agg({
    'subscribers_for_last_30_days': 'sum',
    'Unemployment rate': 'first'
}).reset_index()

# Correlation
corr = subs_unemp_agg['subscribers_for_last_30_days'].corr(subs_unemp_agg['Unemployment rate'])
print(f" Correlation between subscribers gained and unemployment rate: {corr:.2f}")

# Scatter Plot
plt.figure(figsize=(10,6))
sns.scatterplot(data=subs_unemp_agg, x='Unemployment rate', y='subscribers_for_last_30_days', s=100, color='purple')
plt.title('Subscribers Gained vs Unemployment Rate')
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Subscribers Gained (Last 30 Days)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, square=True)

plt.title('Correlation Heatmap:\nSubscribers Gained vs Unemployment Rate', fontsize=14)
plt.tight_layout()
plt.show()

#####################################################################################################

dfvideo_views = df["video_views_for_the_last_30_days"].dropna()
plt.figure(figsize=(14,8))
sns.boxplot(data=df, x='channel_type', y='video_views_for_the_last_30_days', palette='coolwarm')
plt.title('Video Views in Last 30 Days by Channel Type')
plt.xlabel('Channel Type')
plt.ylabel('Video Views (Last 30 Days)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

######################################################################################################

upload_by_month = df.groupby('created_month')['uploads'].mean()

plt.figure(figsize=(10,6))
upload_by_month.plot(kind='bar', color='orange')

plt.title('Average Number of Uploads by Channel Creation Month')
plt.xlabel('Creation Month')
plt.ylabel('Average Number of Uploads')
plt.tight_layout()
plt.show()


#####################################################################################################

# Today's date
today = pd.to_datetime('today')

# Calculate channel age in months
df['channel_age_months'] = ((today - df['created_full_date']).dt.days / 30).clip(lower=1)  # Avoid division by zero

# Average subscribers per month
df['subs_per_month'] = df['subscribers'] / df['channel_age_months']

# Plot distribution
plt.figure(figsize=(12,6))
sns.histplot(df['subs_per_month'], bins=50, color='teal', kde=True)
plt.title('Distribution of Average Subscribers Gained Per Month')
plt.xlabel('Subscribers Gained Per Month')
plt.ylabel('Number of Channels')
plt.tight_layout()
plt.show()

# Optional: Top 10 Channels by subs per month
top_gainers = df[['Youtuber', 'subs_per_month']].sort_values(by='subs_per_month', ascending=False).head(10)
print(top_gainers)

plt.figure(figsize=(12,6))
sns.barplot(data=top_gainers, x='Youtuber', y='subs_per_month', palette='crest')
plt.title('Top 10 Fastest Growing YouTube Channels (Subscribers Gained per Month)')
plt.xlabel('Channel Name')
plt.ylabel('Average Subscribers Gained per Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

##################################################################################################

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Shreyoshi\\OneDrive\\Desktop\\DsResearch\\Media and Technology\\Media and Technology\\Global YouTube Statistics.csv'